# Module 09 — Lecture: Seismic catalogues. Catalogue preprocessing (II)

**DIGIHAZ PhD Course** | Disaster Risk Reduction

---


## Learning Objectives

By the end of this session you will be able to:

1. Represent catalogue properties
2. Decluster catalogues.
3. Calculate the seismic parameters (Gutenberg-Richter law) of the catalogue.

## Introduction

Once the catalogue has been homogenised then all the events can be studied jointly and compared. In this way we have expanded our database, as more events can be used to analyse the seimsicity in the area at the same time. Let us represent the catalogue and observe how the seismicity changes over time:

In [ ]:
pip install pandas

In [ ]:
pip install matplotlib

In [ ]:
import pandas as pd                    # Standard data analysis library.
import numpy as np                     # Vectorized calculus library.
import matplotlib.pyplot as plt        # Data representation library.
import math                            # Math functions library.

After importing the libraries, the next step is to recover the results of the previous lesson and load then with *Pandas*.

In [ ]:
data = pd.read_csv('catalogue_2.csv')
display(data) # Then we can display the data as a table.

We can drop all the columns which are not needed for this analysis and just keep the ones with the moment magnitude, date, time, longitude, latitude, decimal year, depth and ID. We can do this by creating a list containing the names of the columns to keep:
    

In [ ]:
columns = ["Event", "Date", "Time", "Latitude","Longitude","Depth","mw","DecimalYear"]
data = data[columns]
# By using data[list of names] we filter the DataFrame so it only contains the
# required columns.
display(data)

Now let us represent our data by starting with the magnitude of the earthquakes over time:

In [ ]:
fig, axs = plt.subplots(ncols=1, nrows=1)

axs.scatter(data.DecimalYear, data.mw, color="black", s=5)
axs.set_xlabel("Time [year]")
axs.set_ylabel("Magnitude [Mw]")
axs.set_title("Southern Spain Seismic Catalogue")

**Proposed exercise 1) Modify the plot so the colour changes depending on the magnitude value. Use colormaps of your choice to reflect these changes.**


**Proposed exercise 2) Represent the Moment magnitude - Depth distribution (instead of Moment magnitude - Time distribution).**

We can see that the number of earthquakes per year has increased over time, being very scarce in the preinstrumental era. We can zoom-in to see this effect.

In [ ]:
fig, axs = plt.subplots(ncols=1, nrows=1)

axs.scatter(data.DecimalYear, data.mw, color="black", s=5)
axs.set_xlabel("Time [year]")
axs.set_ylabel("Magnitude [Mw]")
axs.set_title("Southern Spain Seismic Catalogue")
axs.set_xlim([1950, 2026])

This effect prevents us to use the whole catalogue in the seismic parameter computation procedure. We will then limit the data so our catalogue starts from 1970.

In [ ]:
data = data[data.DecimalYear >= 1970]
# In this case the filtering has the format:
# data[column >= condition], we could use other comparisons such as
# <= < > == or !=

# Since the minimum magnitude value approaches 3.0 except for 4 events, let us
# filter the catalogue so these for events are not shown.
data = data[data.mw >= 3.0]
data.reset_index(inplace=True, drop=True)

Once the catalogue has been filtered we can approach the computation of the seimsic parameters as defined in the Gutenberg-Richter law.

## Gutenberg-Richter law (G-R law)

This semiempirical law was proposed by Beno Gutenberg and Charles Francis Richter in 1944 after carefully studying the seismicity in California. It was latter used to analyse the earthquakes in other regions, confirming its applicability outside the region of definition.

This law states that the frequency of the earthquakes decreases exponentially with the energy they release. The bigger the earthquake, the lower that chances for it to occur. Normally, it is expressed with the following equation:

${\displaystyle \log _{10}N=a-bM}$

where $N$ is the number of earthquakes with magnitude greater or equal to $M$, $a$ is the seismic productivity (ordinate in the origin) and $b$ is related to the ratio between small earthquakes and big earthquakes (slope). Normally, the b-value in a global catalogue is around 1. This parameter is of great interest in research, as it has been related with the tectonic stress build-up, and thus, regarded as a seismicity forecast indicator (decreases of the b-value time series are related with an increased probability of having bigger earthquakes).

There are several ways of computing the seismic parameters, for instance, one of the most used algorithms is based on the Aki-Utsu (Aki, 1965; Utsu, 1966) formula:

b$ = \frac{\log_{10} e}{\overline{M} + \Delta M * 0.5 - M_{c}}$

where $\overline{M}$ is the mean magnitude of the catalogue $\Delta M$ is the binning of the catalogue (normally set as 0.1) and $M_{c}$ is the completeness magnitude of the catalogue.

With related uncertainty:

$\sigma_b = \frac{b}{\sqrt{N}}$
where N is the number of earthquakes.

It can be seen that one of the most critical parameters is the completeness magnitude. Before introducing the script for the computation of such parameter let us represent the inverse cumulative distribution of the seismicity.

This representation is the one used to fit the G-R law to the catalogue data. It is constructed by asigning to each bin (normally from 0.1 to 0.1 in Mw) the number of events with magnitude equal or greater than said bin value. For example, in the bin 3.1 we would expect the number of earhtquakes with magnitude equal or greater than 3.1, and so on for the rest of bins.

In [ ]:
# First we obtain the bins in which we will classify the earthquakes.
# For that we take the minimum value of the magnitude in the catalogue
# and the maximum value of the catalogue (rounded to 1 decimal place)
# then we input 0.1 as the binning.
n_bins = np.arange(np.round(data.mw.min(), 1),
                   np.round(data.mw.max(), 1) + 0.1, 0.1)
# With the np.histogram function we compute the number of earthquakes in
# each bin and then we sum them from last to first [::-1]
# Then we invert this distribution so it outputs the desired value for
# each bin (N(m >= M)), where M is the magnitude of the bin.
counts, bins = np.histogram(data.mw, n_bins)
ccounts = np.cumsum(counts[::-1])
iccounts = ccounts[::-1]
# Let us plot the results. We will apply the base 10
# logratithm on to the cumulative counts to linearize the relation

fig2, axs2 = plt.subplots(nrows=1, ncols=1)

axs2.scatter(bins[:-1], np.log10(iccounts), color="black", s=5)
axs2.set_title("Inverse cumulative distribution of seismicity")
axs2.set_xlabel("Magnitude [Mw]")
axs2.set_ylabel("$log_{10}(N_{m ≥ M})$")

We could now just fit the straight line (**y = mx + n**) that best describes the data, either graphically or with any algorithm. Instead we will make use of one the most used scripts from ZMAP (Wiemer, 2002), the *McBest* algortihm. originally written for the MATLAB programming language. 

In [ ]:
def _inclusive_arange(start: float, stop: float, step: float) -> np.ndarray:
    # Modification of the np.arange so it includes the border values.
    n = int(round((stop - start) / step))
    if n < 0:
        return np.array([], dtype=float)
    return start + step * np.arange(n + 1, dtype=float)

def haversine(lon1: np.ndarray, lat1: np.ndarray,
              lon2: np.ndarray, lat2: np.ndarray) -> np.ndarray:
    R = 6371.0  # Earth radius in km

    # Convert to radians
    lon1_rad = np.radians(lon1)
    lat1_rad = np.radians(lat1)
    lon2_rad = np.radians(lon2)
    lat2_rad = np.radians(lat2)

    # Create pairwise differences (A vs B)
    dlat = lat1_rad[:, None] - lat2_rad
    dlon = lon1_rad[:, None] - lon2_rad

    # Haversine formula
    a = (np.sin(dlat / 2)**2 +
         np.cos(lat1_rad[:, None]) *
         np.cos(lat2_rad) *
         np.sin(dlon / 2)**2)

    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    return R * c

def calc_bmemag(magnitudes: np.ndarray,
                bin_interval: float = 0.1) -> tuple[float, float, float]:
    mags = np.asarray(magnitudes, dtype=float)
    mags = mags[np.isfinite(mags)]
    n = mags.size
    if n == 0:
        return (math.nan, math.nan, math.nan)

    min_mag = float(np.min(mags))
    mean_mag = float(np.mean(mags))

    denom = mean_mag + 0.5 * bin_interval - min_mag
    if denom <= 0:
        return (math.nan, math.nan, math.nan)

    b_value = (1.0 / denom) * math.log10(math.e)

    if n < 2:
        b_std = math.nan
    else:
        b_variance_by_n = float(np.var(mags - mean_mag, ddof=1) / n)
        b_std = 2.30 * math.sqrt(max(0.0, b_variance_by_n)) * (b_value**2)

    a_value = math.log10(n) + b_value * min_mag
    return (b_value, b_std, a_value)

def _do_calculation(these_mags: np.ndarray, bin_interval: float,
                    hypothetical_mc: float) -> float:
    b_value, _, _ = calc_bmemag(these_mags, bin_interval)
    if not np.isfinite(b_value):
        return math.nan

    half_bin = 0.5 * bin_interval
    vmag = _inclusive_arange(hypothetical_mc, 15.0, bin_interval)
    if vmag.size == 0:
        return math.nan

    vnumber = 10.0 ** (math.log10(these_mags.size) - b_value 
                       * (vmag - hypothetical_mc))
    vnumber = np.round(vnumber)

    pmedges = np.concatenate([vmag - half_bin, np.array([vmag[-1] + half_bin])])
    bval = np.histogram(these_mags, pmedges)[0].astype(float)
    b3 = np.cumsum(bval[::-1])[::-1]
    total = float(np.sum(b3))
    if total <= 0:
        return math.nan

    return float(np.sum(np.abs(b3 - vnumber)) / total * 100.0)

def _max_curvature_seed(magnitudes: np.ndarray, bin_interval: float) -> float:
    half_bin = 0.5 * bin_interval
    centers = _inclusive_arange(-2.0, 6.0, bin_interval)
    edges = np.concatenate([centers - half_bin,
                            np.array([centers[-1] + half_bin])])
    hist = np.histogram(magnitudes, edges)[0].astype(float)
    if hist.size == 0:
        return math.nan

    max_count = np.max(hist)
    idx_last = int(np.where(hist == max_count)[0][-1])
    return float(centers[idx_last])

def calc_mc_best(magnitudes: np.ndarray,
                 bin_interval: float = 0.1) -> tuple[float, float, float]:
    # calc_McBest from ZMAP scripts:
    mags = np.asarray(magnitudes, dtype=float)
    mags = np.sort(mags[np.isfinite(mags)])
    if mags.size == 0:
        return (math.nan, math.nan, math.nan)

    mc_start = _max_curvature_seed(mags, bin_interval)
    if not np.isfinite(mc_start):
        return (math.nan, math.nan, math.nan)

    half_bin = 0.5 * bin_interval
    mag_centers = _inclusive_arange(mc_start - 0.9, mc_start + 1.5, bin_interval)
    edges = np.concatenate([mag_centers - half_bin,
                            np.array([mag_centers[-1] + half_bin])])

    mags_desc = mags[::-1]
    n_gt_edge = np.array([np.sum(mags_desc > edge) for edge in edges[:-1]],
                         dtype=int)
    too_few = n_gt_edge < 25

    results = np.full(mag_centers.shape, np.nan, dtype=float)
    for i, hyp_mc in enumerate(mag_centers):
        if too_few[i]:
            continue
        n_use = int(n_gt_edge[i])
        results[i] = _do_calculation(mags_desc[:n_use], bin_interval, float(hyp_mc))

    def first_center(threshold: float) -> float:
        idx = np.where(results < threshold)[0]
        if idx.size == 0:
            return math.nan
        return float(mag_centers[idx[0]])

    mc90 = first_center(10.0)
    mc95 = first_center(5.0)

    mc_best = math.nan
    for thr in (10.0, 15.0, 20.0, 25.0):
        candidate = first_center(thr)
        if np.isfinite(candidate):
            mc_best = candidate
            break

    return (mc_best, mc95, mc90)

Then we call the function by providing just the magnitudes of the catalogue:

In [ ]:
mc = calc_mc_best(data.mw)

In [ ]:
display(mc)

The best completeness magnitude value is then the first one in tuple with three elements mc[0]. We can use this value to compute the seismic parameters of the catalogue (namely the a and b-value of the G-R law)

In [ ]:
def seism_params(magnitudes:np.array([], dtype=float),
                 mc:float, deltaM:float=0.1) -> np.array:
    bval = (np.log10(np.exp(1)) /
            (np.mean(magnitudes) + deltaM * 0.5 - mc))
    n = len(magnitudes)
    aval = np.log10(n) + bval * mc
    
    return aval, bval

In [ ]:
s = seism_params(data.mw, mc[0])
display(s)

We have obtained a rather high b-value (1.428). This can be explained by both a surplus in the 3.0-3.5 range of the catalogue and 4.5-5.0 that can be seen in the inverse cumulative distribution (we can see two plateaus).

## Declustering

This surplus can be related to the clustered seismicity, i.e. events that belong to a seismic serie or swarm. The seismic series are composed mainly by the mainshocks and the aftershocks, and some of them also include foreshocks (earthquakes that precede the main event of the cluster). In general, it is expected that the mainshock is the most intense event of the cluster, and that in average the biggest aftershock at most 1.2 M less than the main shock. This is called the Båth’s law (Båth, 1965). Nevertheless, the existence of swarms contradicts this law, as several big earthquakes can be included in the clusters identified as such.

In this section we will decluster the catalogue, so only the independent events are considered in the seismic parameter computation. This step is required if the catalogue is to be used in the long-term seismic hazard computation. The declustering procedure ensures that the catalogue follows a Poisson distribution (i.e. a random point process with constant rate $\lambda$ can be used to describe the seismicity during a period of time in a certain location).

There are several algorithms used in declustering procedures. They can be classified into families:

1) Window algorithms
2) Stochastic algorithms
3) Genetic/Correlation algorithms

Window algorithms are the easiest to implement out of the three families, and require less computation power. Openquake's script suite (Pagani, 2014) has three different window declustering methods (the only difference between them is the functions used to compute the spatio-temporal windows): Reasenberg-Jones (), Urhammer () and Gardner-Knopoff ().

The following script contains the different time and space windows definitions':

In [ ]:
def time_window_cutoff(sw_time:np.array(dtype=float),
                       time_cutoff:float, days: float = 60.0) -> np.ndarray:
    """
    Cut-off values for time windows (optional)

    Parámetros
    ----------
    sw_time : ARRAY, FLOAT
        Time windows values.
    time_cutoff : FLOAT
        Maximum number of days between one event and the following inside a cluster.
    days : FLOAT
        Model parameter for number of days in foreshock sequence. 60 days default.

    Returns
    -------
    sw_time : ARRAY
        Modified time window using the cutoff times.
    """

    sw_time = np.array(
        [(time_cutoff / days) if x > (time_cutoff / days)
            else x for x in sw_time])
    return sw_time

def spatial_window_GK(magnitude:np.array([], dtype=float)) -> np.ndarray:
    """
    Spatial window from Gardner-Knopoff algorithm.

    Parameters
    ----------
    magnitude : FLOAT, ARRAY
        Magnitudes (mw) of the catalogue.

    Returns
    -------
    distance_windows : FLOAT, ARRAY
        Distance from one earthquake to the rest using the expression
        10^(0.1238*Mw+0.983)

    """

    distance_windows = np.power(10.0, 0.1238 * magnitude + 0.983)
    
    return distance_windows

def time_window_GK(magnitude:np.array([], dtype=float),
                   time_cutoff:float=0.0, days:float=60.0) -> np.ndarray:
    """
    Time windows from Gardner-Knopoff.

    Parameters
    ----------
    magnitude : FLOAT, ARRAY
        magnitudes from the catalogue.
    time_cutoff : FLOAT
        Maximum number of days between events. Applied only if > 0.
    days : FLOAT
        Model parameter for number of days in foreshock sequence. 60 days default.

    Returns
    -------
    time_windows : FLOAT, ARRAY
        Cálculo de "distancia temporal" entre un evento y el resto usando la 
        expresión 10^(0.032 * Mw + 2.7389)/days si Mw >= 6.5 y 
        10^(0.5409 * Mw - 0.547) / days si Mw < 6.5

    """
    time_windows = np.power(10.0, 0.032 * magnitude + 2.7389) / days
    mask = magnitude < 6.5  # Boolean filtering for Mw < 6.5
    time_windows[mask] = np.power(10.0, 0.5409 * magnitude[mask] - 0.547) / days
    
    if time_cutoff > 0:
        time_windows = time_window_cutoff(time_windows, time_cutoff, days)
    
    return time_windows


Before defining the main loop of the catalogue several parameters and constants must be set:

In [ ]:
n_eq = len(data)    # Number of events in the catalogue.
decimal_year = data.DecimalYear.to_numpy()   
magnitudes = data.mw.to_numpy()

# Time window parameters.

tp = 1.0                # Proper time.
tw_p = 60.0             # Time window parameter.
ct = 0                  # Maximum time foreshocks/aftershocks.

sw = spatial_window_GK(magnitudes)             # Spatial window.
tw = time_window_GK(magnitudes, ct, tw_p)      # Time window.

eq_id = data.index.to_numpy()                   # Earthquake_ID
vcl = np.zeros(n_eq, dtype=int)                # Cluster_ID.
# Rearranging earthquakes from smallest to biggest.
id0 = np.flipud(np.argsort(magnitudes, kind="heapsort"))
longitudes = data['Longitude'][id0].to_numpy()
latitudes = data['Latitude'][id0].to_numpy()
# Rearranging all list using the new order.
sw_space = sw[id0]        
sw_time = tw[id0]       
year_dec = decimal_year[id0]   
eqid = eq_id[id0]
flagvector = np.zeros(n_eq, dtype=int)  # Type of event.

## If flagvector == 0 : Main shock.
## If flagvector == 1 : Aftershock.
## iF flagvecror == -1: Foreshock.

Now we can proceed with the main loop of the declustering process. We will obtain a flag column which informs about whether an earthquake is a main shock, a foreshock or an aftershock.

In [ ]:
# Main loop.

clust_index = 0       # Cluster ID.
for i in range(0, n_eq - 1):
    if vcl[i] == 0:
        # We look for events inside the defined time window.
        dt = year_dec - year_dec[i]
        vsel = np.logical_and(
            vcl == 0,
            np.logical_and(
                dt >= (-sw_time[i] * tp),
                dt <= sw_time[i]))
        # From all the events inside the time windows, we search for
        # those inside the spatial window.
        vsel1 = haversine(longitudes[vsel],
                          latitudes[vsel],
                          longitudes[i],
                          latitudes[i]) <= sw_space[i]
        vsel[vsel] = vsel1[:,0]
        temp_vsel = np.copy(vsel)
        temp_vsel[i] = False
        
        if any(temp_vsel):
            # We assing the cluster index.
            vcl[vsel] = clust_index + 1
            flagvector[vsel] = 1
            # Those events inside the cluster that precede the 
            # mainshock, are assigned -1 in flagvector (foreshock)
            temp_vsel[dt >= 0.0] = False
            flagvector[temp_vsel] = -1
            flagvector[i] = 0
            clust_index += 1

After the main loop is executed we must retrieve the cluster ID information and the flagvector (type of earthquake: mainshock, foreshock or aftershock).

In [ ]:
# The lists must be returned to its original order.
id1 = np.argsort(eqid, kind='heapsort')
eqid = eqid[id1]
vcl = vcl[id1]
flagvector = flagvector[id1]
# The results are embedded into the original catalogue.
data['Flag'] = flagvector
data['Cluster_label'] = vcl

Now we can filter the catalogue so only the mainshocks are considered in the analysis:

In [ ]:
data_dec = data[data.Flag == 0]

Let us plot the cumulative distribution again and then compute the seismic parameters, so we can compare with the non-declustered catalogue:

In [ ]:
n_bins2 = np.arange(np.round(data_dec.mw.min(), 1),
                   np.round(data_dec.mw.max(), 1) + 0.1, 0.1)

counts2, bins2 = np.histogram(data_dec.mw, n_bins2)
ccounts2 = np.cumsum(counts2[::-1])
iccounts2 = ccounts2[::-1]

fig3, axs3 = plt.subplots(nrows=1, ncols=1)

axs3.scatter(bins2[:-1], np.log10(iccounts2), color="black", s=5)
axs3.set_title("Inverse cumulative distribution of seismicity")
axs3.set_xlabel("Magnitude [Mw]")
axs3.set_ylabel("$log_{10}(N_{m ≥ M})$")

We can see how the distribution can be fitted to a straight line to the range 3 to 5 Mw. The declustering procedure increases the quality of the G-R law fit.

In [ ]:
mc_2 = calc_mc_best(data_dec.mw)
s_2 = seism_params(data_dec.mw, mc_2[0])
display(s_2)

The computed b-value is lower than in the non-declustered catalogue (1.43) and closer to what is expected for the region (from 1.03 to 1.10).

## Conclusions and exercises

In this lesson we have plotted the catalogue to analyse both the catalogue time serie and the frequency-energy distribution. With this information we obtained the seismic parameters, but we noticed excess in some ranges of the distribution. A declustering process might correct the first seismic parameters computation. After the declustering process the seismic parameters are closer to the background values obtained in long-term studies.

The following exercises are left for the reader:

1) Using the seismic parameters obtained in the computations, can you provide the expected number of earthquakes with magnitude greater than 5.0? Is it close to the real number? HINT: Definition of the Gutenberg-Richter law.
2) How many aftershocks, foreshocks and mainshocks are identfied in the catalogue? Create a table that shows both the absolute number and percentage.
3) Create a plot that shows both the non-declustered and declustered catalogue frequency-energy distribution (inverse cumulative distribution). Use different colours and use a distinct marker to indicate the completeness magnitude. Extra: try and plot the G-R fit over each inverse cumulative distribution using a solid line.

With these two lessons we have studied how to prepare the catalogues for their use towards research. This step is critical and may involve complex analysis but, nevertheless, should also be included in any scientific communication. First, because it enables reproducibility and second, because not all the regions or studies require the same steps inside the catalogue preparation stage so it is not obvious.